In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm

import pystac

import rioxarray as rxr
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
from helpers.shared import assert_grid_match

In [ ]:
from config import *

# -- Catalog ----------------------------------------------------------------------------------------------------------------------------------------------------------------------
CATALOG             = load_catalog()
INPUT_ASSET         = "reproj"

# -- Output -----------------------------------------------------------------------------------------------------------------------------------------------------------------------
OUT_DIR             = Path("../out/clip")
MKDIR               = True
OUT_NAME            = "clip"
OUT_EXT             = "tif"

# -- Execution Flags -------------------------------------------------------------------------
OVERWRITE                   = False      # Whether the output should be overwritten

In [ ]:
wv2_col = CATALOG.get_child("wv2-imagery")
sd8_col = CATALOG.get_child("sd8-imagery")
items   = list(wv2_col.get_items()) + list(sd8_col.get_items())

tqdm.write(f"{len(items)} item(s) found:")
for item in items:
    tqdm.write(f"ID: {item.id} | Date: {item.datetime} | Path: {item.assets[INPUT_ASSET].href}")

In [ ]:
OUT_DIR.mkdir(parents = True, exist_ok=MKDIR)

In [ ]:
def write_clip(out_path, da, profile_options):

    n_bands, h, w = da.shape
    crs = da.rio.crs
    transform = da.rio.transform()

    profile = dict(
        height = h,
        width = w,
        count = n_bands,
        crs = crs,
        transform = transform,
        **profile_options
    )

    with rasterio.open(out_path, "w", **profile) as dst:
        bands = list(da.groupby("band"))
        pbar_band = tqdm(total=len(bands), desc = "Writing band", unit = "band", leave = True)
        with pbar_band:
            for i, (_, block) in enumerate(bands, start = 1):
                pbar_band.set_description(f"Writing band {i+1}")
                dst.write(block.values.squeeze(), i)
                pbar_band.update(1)

    with rasterio.open(out_path, "r+") as dst:
        pbar_ovr = tqdm(total = 1, desc = "Building overviews", leave = True)
        with pbar_ovr:
            dst.build_overviews([2, 4, 8, 16], Resampling.nearest)
            dst.update_tags(ns="rio_overview", resampling = "nearest")
            pbar_ovr.update(1)

In [ ]:
pbar = tqdm(enumerate(items, start=1), unit="scene", desc="Clipping", dynamic_ncols=True, leave=True, total=len(items))
for i, item in pbar:
    pbar.set_description(f"[{i}/{len(items)}] Clipping {item.id}")

    out_path = OUT_DIR / f"{item.id}-{OUT_NAME}.{OUT_EXT}"

    # Update catalog
    asset = pystac.Asset(
        href=str(out_path.resolve()),
        media_type=pystac.MediaType.GEOTIFF,
        roles=["data"],
        title = "Clipped Scene",
    )

    item.add_asset(OUT_NAME, asset)

    # Check if overwriting
    if out_path.exists() and not OVERWRITE:
        tqdm.write(f"Skipping (exists): {out_path.name}")
        continue

    # Clip
    da = rxr.open_rasterio(item.assets[INPUT_ASSET].href, chunks = CHUNKS)
    box = gpd.read_file(item.assets["common-footprint"].href).to_crs(da.rio.crs)
    minx, miny, maxx, maxy = box.total_bounds
    da = da.rio.clip_box(minx = minx, miny = miny, maxx = maxx, maxy = maxy)
    
    # Write
    write_clip(
        out_path=out_path,
        da=da,
        profile_options=PROFILE_FLOAT32)

    tqdm.write(f"Saved: {out_path.name}")

CATALOG.normalize_hrefs(str(STAC_DIR))
CATALOG.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
tqdm.write("Complete")

In [ ]:
assert_grid_match(*[item.assets[OUT_NAME].href for item in items])